In [2]:
import pandas as pd


In [9]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(
    "../data/raw/complaint_embeddings.parquet"
)

print("Rows:", parquet_file.metadata.num_rows)
print("Row groups:", parquet_file.num_row_groups)
print("Columns:", parquet_file.schema.names)

Rows: 1375327
Row groups: 2
Columns: ['id', 'document', 'element', 'chunk_index', 'company', 'complaint_id', 'date_received', 'issue', 'product', 'product_category', 'state', 'sub_issue', 'total_chunks']


In [13]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("../data/raw/complaint_embeddings.parquet")

for batch in parquet_file.iter_batches(
    batch_size=5,
    columns=["document"]
):
    df = batch.to_pandas()
    print(df.head())
    break

                                            document
0  a card was opened under my name by a fraudster...
1  i made the mistake of using my wellsfargo debi...
2  and got a letter stating my dispute was reject...
3  dear cfpb, i have a secured credit card with c...
4  y confirmation whatsoever to report to the pol...


In [14]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [16]:
index = faiss.read_index("../vector_store/complaints_faiss.index")

metadata_df = pd.read_parquet("../vector_store/chunks_metadata.parquet")

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 623.60it/s]


In [17]:
def retrieve_chunks(query, k=5):
    """
    Takes a user question and returns top-k relevant complaint chunks
    """

    # Step 1: Convert query to embedding
    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding,
        dtype=np.float32
    )

    # Step 2: Search FAISS
    distances, indices = index.search(query_embedding, k)

    # Step 3: Get results from metadata
    results = metadata_df.iloc[indices[0]].copy()

    # Step 4: Attach similarity scores
    results["distance"] = distances[0]

    return results

In [21]:
results = retrieve_chunks("unauthorized credit card charges")

for _, row in results.iterrows():
    print("\n" + "="*80)
    print("Product:", row["product"])
    print("Issue:", row["issue"])
    print("Company:", row["company"])
    print("Distance:", row["distance"])



Product: Credit card
Issue: Fees or interest
Company: JPMORGAN CHASE & CO.
Distance: 0.440578818321228

Product: Credit card
Issue: Problem with a purchase shown on your statement
Company: BANK OF AMERICA, NATIONAL ASSOCIATION
Distance: 0.5077540874481201

Product: Money transfer, virtual currency, or money service
Issue: Unauthorized transactions or other transaction problem
Company: Block, Inc.
Distance: 0.5201938152313232

Product: Credit card
Issue: Problem with a purchase shown on your statement
Company: KEYCORP
Distance: 0.5246365070343018

Product: Credit card
Issue: Other features, terms, or problems
Company: ENOVA INTERNATIONAL, INC.
Distance: 0.5461913347244263


A retrieval function was implemented that converts user queries into embeddings using SentenceTransformer, searches the FAISS index for nearest neighbors, and returns the top-k most relevant complaint chunks along with their metadata.

PROMPT_TEMPLATE
You are a helpful and professional financial complaint analyst working for CrediTrust.

Your job is to analyze customer complaint excerpts and answer the user's question clearly and accurately.

Rules:
- Use ONLY the provided context below.
- Do NOT use external knowledge.
- If the context does not contain enough information, say: "The provided context does not contain enough information to answer this question."
- Be concise, clear, and factual.
- Focus on patterns in customer complaints.

Context:
{context}

Question:
{question}

Answer:
"""